In [1]:
pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import datetime
import requests
import json
import os

from dotenv import load_dotenv
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text, MetaData, Table
from sqlalchemy.orm import sessionmaker


In [ ]:
Завдання 1: Оновлення інформації про клієнта (2 бали)
Створіть функцію для оновлення контактної інформації клієнта за його номером з наступними можливостями:

Оновлення телефону клієнта
Оновлення email (якщо поле існує в таблиці)
Опціонально, якщо вам хочеться більше практики:

Логування змін в окрему таблицю
Використайте підхід з параметризованими запитами через text() та UPDATE оператор. 
    Не забудьте на початку перевірити чи існує клієнт з таким номером в базі - це хороша практика.

Отримати всі колонки, які існують в таблиці ви можете наступним запитом

  SELECT COLUMN_NAME, DATA_TYPE
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_NAME = 'customers'
Запустіть функцію і продемонструйте її роботу, запустивши SELECT, який допоможе це зробити.

In [6]:
import datetime
import pandas as pd
from sqlalchemy import text

# 1. Які колонки є в таблиці customers (фільтруємо за поточною БД)
columns_df = pd.read_sql(text("""
    SELECT COLUMN_NAME, DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = 'customers'
"""), engine)
columns_df

,COLUMN_NAME,DATA_TYPE
0,customerNumber,int
1,customerName,varchar
2,contactLastName,varchar
3,contactFirstName,varchar
4,phone,varchar
5,addressLine1,varchar
6,addressLine2,varchar
7,city,varchar
8,state,varchar
9,postalCode,varchar


In [7]:
def update_customer_contact(engine, customer_number, new_phone=None, new_email=None):
    """
    Оновлює телефон та/або email клієнта в ОДНІЙ транзакції
    з перевіркою існування клієнта і логуванням змін.
    """
    if new_phone is None and new_email is None:
        print("⚠️ Не передано жодного поля для оновлення")
        return False

    try:
        # engine.begin() = автоматичний COMMIT при успіху і ROLLBACK при помилці
        with engine.begin() as conn:
            # Крок 1: чи існує клієнт (і поточні значення)
            customer = conn.execute(
                text("SELECT * FROM customers WHERE customerNumber = :num"),
                {"num": customer_number}
            ).mappings().first()

            if customer is None:
                print(f"❌ Клієнта з номером {customer_number} не знайдено")
                return False

            # Крок 2: чи існує поле email у таблиці
            email_exists = conn.execute(text("""
                SELECT COUNT(*) FROM INFORMATION_SCHEMA.COLUMNS
                WHERE TABLE_SCHEMA = DATABASE()
                  AND TABLE_NAME = 'customers' AND COLUMN_NAME = 'email'
            """)).scalar() > 0

            now = datetime.datetime.now()
            log_query = text("""
                INSERT INTO customer_change_log
                    (customerNumber, field_name, old_value, new_value, changed_at)
                VALUES (:num, :field, :old, :new, :now)
            """)

            # Крок 3: оновлення телефону
            if new_phone is not None:
                conn.execute(
                    text("UPDATE customers SET phone = :phone WHERE customerNumber = :num"),
                    {"phone": new_phone, "num": customer_number}
                )
                conn.execute(log_query, {"num": customer_number, "field": "phone",
                                         "old": customer["phone"], "new": new_phone, "now": now})
                print(f"✅ Телефон оновлено: {customer['phone']} -> {new_phone}")

            # Крок 4: оновлення email (лише якщо колонка існує)
            if new_email is not None:
                if email_exists:
                    conn.execute(
                        text("UPDATE customers SET email = :email WHERE customerNumber = :num"),
                        {"email": new_email, "num": customer_number}
                    )
                    conn.execute(log_query, {"num": customer_number, "field": "email",
                                             "old": customer.get("email"), "new": new_email, "now": now})
                    print(f"✅ Email оновлено на {new_email}")
                else:
                    print("⚠️ Поле email відсутнє в таблиці customers, оновлення пропущено")

        return True

    except Exception as e:
        print(f"❌ Помилка: {e}")
        print("🔄 Усі зміни скасовано (ROLLBACK)")
        return False

In [9]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS customer_change_log (
            log_id INT AUTO_INCREMENT PRIMARY KEY,
            customerNumber INT NOT NULL,
            field_name VARCHAR(50) NOT NULL,
            old_value VARCHAR(255),
            new_value VARCHAR(255),
            changed_at DATETIME NOT NULL
        )
    """))

In [10]:
pd.read_sql(text("SHOW TABLES LIKE 'customer_change_log'"), engine)

,Tables_in_classicmodels (customer_change_log)
0,customer_change_log


In [11]:
check_query = text("""
    SELECT customerNumber, customerName, phone
    FROM customers WHERE customerNumber = :num
""")

print("До оновлення:")
display(pd.read_sql(check_query, engine, params={"num": 103}))

update_customer_contact(engine, 103, new_phone="40.32.9999", new_email="test@example.com")

print("Після оновлення:")
display(pd.read_sql(check_query, engine, params={"num": 103}))

print("Лог змін:")
display(pd.read_sql(text("SELECT * FROM customer_change_log ORDER BY log_id DESC"), engine))

# Неіснуючий клієнт
update_customer_contact(engine, 99999, new_phone="000")

До оновлення:


,customerNumber,customerName,phone
0,103,Atelier graphique,40.32.2555


✅ Телефон оновлено: 40.32.2555 -> 40.32.9999
⚠️ Поле email відсутнє в таблиці customers, оновлення пропущено
Після оновлення:


,customerNumber,customerName,phone
0,103,Atelier graphique,40.32.9999


Лог змін:


,log_id,customerNumber,field_name,old_value,new_value,changed_at
0,1,103,phone,40.32.2555,40.32.9999,2026-10-04 20:10:42


❌ Клієнта з номером 99999 не знайдено


False

In [ ]:
Завдання 2: Створення нового замовлення з транзакцією (5 балів)
Реалізуйте процес створення нового замовлення з наступними кроками в одній транзакції:

Створення запису в таблиці orders
Додавання товарних позицій в orderdetails
Перевірка наявності товарів на складі
Зменшення кількості товарів на складі
Запустіть процес з тестовими даними і продемонструйте через SELECT,
що процес успішно відпрацював і були виконані необхідні операції.

In [13]:
def create_order(engine, customer_number, items, required_days=7, comments=None):
    """
    Створює замовлення в ОДНІЙ транзакції.
    items: список словників {"productCode": "S10_1678", "quantity": 5, "price": None}
           (якщо price=None, береться MSRP з таблиці products)
    """
    try:
        with engine.begin() as conn:  # COMMIT при успіху, ROLLBACK при помилці
            # Крок 0: чи існує клієнт
            exists = conn.execute(
                text("SELECT COUNT(*) FROM customers WHERE customerNumber = :c"),
                {"c": customer_number}
            ).scalar()
            if not exists:
                raise ValueError(f"Клієнта {customer_number} не знайдено")

            # Крок 1: новий номер замовлення і запис у orders
            new_order_number = conn.execute(
                text("SELECT COALESCE(MAX(orderNumber), 0) + 1 FROM orders FOR UPDATE")
            ).scalar()

            today = datetime.date.today()
            conn.execute(text("""
                INSERT INTO orders
                    (orderNumber, orderDate, requiredDate, shippedDate,
                     status, comments, customerNumber)
                VALUES
                    (:num, :odate, :rdate, NULL, 'In Process', :comments, :cust)
            """), {
                "num": new_order_number,
                "odate": today,
                "rdate": today + datetime.timedelta(days=required_days),
                "comments": comments,
                "cust": customer_number,
            })
            print(f"✅ Створено замовлення №{new_order_number}")

            # Кроки 2-4: по кожній позиції
            for line_no, item in enumerate(items, start=1):
                code, qty = item["productCode"], item["quantity"]

                # Крок 3: перевірка наявності (FOR UPDATE блокує рядок до кінця транзакції)
                product = conn.execute(text("""
                    SELECT quantityInStock, MSRP
                    FROM products WHERE productCode = :code FOR UPDATE
                """), {"code": code}).mappings().first()

                if product is None:
                    raise ValueError(f"Товар {code} не існує")
                if product["quantityInStock"] < qty:
                    raise ValueError(
                        f"Недостатньо товару {code}: на складі "
                        f"{product['quantityInStock']}, потрібно {qty}"
                    )

                price = item.get("price") or product["MSRP"]

                # Крок 2: додаємо позицію в orderdetails
                conn.execute(text("""
                    INSERT INTO orderdetails
                        (orderNumber, productCode, quantityOrdered, priceEach, orderLineNumber)
                    VALUES (:num, :code, :qty, :price, :line)
                """), {"num": new_order_number, "code": code,
                       "qty": qty, "price": price, "line": line_no})

                # Крок 4: зменшуємо залишок на складі
                conn.execute(text("""
                    UPDATE products
                    SET quantityInStock = quantityInStock - :qty
                    WHERE productCode = :code
                """), {"qty": qty, "code": code})

                print(f"✅ Позиція {line_no}: {code} x{qty} (склад: "
                      f"{product['quantityInStock']} -> {product['quantityInStock'] - qty})")

        print(f"🎉 Замовлення №{new_order_number} успішно оформлено (COMMIT)")
        return new_order_number

    except Exception as e:
        print(f"❌ Помилка: {e}")
        print("🔄 Усі зміни скасовано (ROLLBACK)")
        return None

#Демонстрація (SELECT до і після):

# Тестові дані: клієнт 103 замовляє два товари
test_items = [
    {"productCode": "S10_1678", "quantity": 5},
    {"productCode": "S10_1949", "quantity": 3},
]

stock_query = text("""
    SELECT productCode, productName, quantityInStock
    FROM products WHERE productCode IN ('S10_1678', 'S10_1949')
""")

print("Склад ДО:")
display(pd.read_sql(stock_query, engine))

order_no = create_order(engine, 103, test_items, comments="Тестове замовлення")

print("Склад ПІСЛЯ:")
display(pd.read_sql(stock_query, engine))

print("Замовлення:")
display(pd.read_sql(text("SELECT * FROM orders WHERE orderNumber = :n"),
                    engine, params={"n": order_no}))

print("Позиції замовлення:")
display(pd.read_sql(text("SELECT * FROM orderdetails WHERE orderNumber = :n"),
                    engine, params={"n": order_no}))

Склад ДО:


,productCode,productName,quantityInStock
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7933
1,S10_1949,1952 Alpine Renault 1300,7305


❌ Помилка: (pymysql.err.DataError) (1366, "Incorrect string value: '\\xD0\\xA2\\xD0\\xB5\\xD1\\x81...' for column 'comments' at row 1")
[SQL: 
                INSERT INTO orders
                    (orderNumber, orderDate, requiredDate, shippedDate,
                     status, comments, customerNumber)
                VALUES
                    (%(num)s, %(odate)s, %(rdate)s, NULL, 'In Process', %(comments)s, %(cust)s)
            ]
[parameters: {'num': 10426, 'odate': datetime.date(2026, 10, 4), 'rdate': datetime.date(2026, 10, 11), 'comments': 'Тестове замовлення', 'cust': 103}]
(Background on this error at: https://sqlalche.me/e/21/9h9h)
🔄 Усі зміни скасовано (ROLLBACK)
Склад ПІСЛЯ:


,productCode,productName,quantityInStock
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7933
1,S10_1949,1952 Alpine Renault 1300,7305


Замовлення:


,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber


Позиції замовлення:


,orderNumber,productCode,quantityOrdered,priceEach,orderLineNumber
